# Generate LLM class descriptions (frozen artifact)

Writes `config/descriptions/{DATASET}_{STYLE}.json` **once**, to be committed
to the repo. Every later notebook (`extract_vlm_features.ipynb`,
`run_al_main.ipynb`) just reads this file -- it is never regenerated as part
of a run.

**This is a frozen artifact, not a reproducible computation.** A hosted
model call is not bit-for-bit reproducible even at `temperature=0.0` --
Google's own docs do not guarantee determinism across requests, let alone
across months as the served weights behind a fixed model name change. What
is reproducible is the **written JSON file** committed to git, not "re-run
this notebook and expect the same text." If a description needs to change,
regenerate deliberately (`OVERWRITE=True`) and commit the new file.

Runs on CPU -- no GPU needed, no dataset image attached. Kept as a notebook
(rather than a local script) only for consistency with every other pipeline
stage in this project.

**One configuration per run** (`DATASET`, `STYLE` are singular, same §2.0
convention as every other notebook here) -- sweeping styles or datasets means
running this notebook again with the EDIT cell changed.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "google-genai"])

if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
# ---- EDIT THIS CELL ----
DATASET = "pathmnist"          # pathmnist | histoset | skintissue
MODEL = "gemini-2.5-flash"     # PINNED. Do not use the gemini-3.x family --
                                # temperature/topK/topP are deprecated and
                                # silently ignored there (see features/descriptions.py).
STYLE = "llm_short"            # llm_short | llm_morphology | llm_multi
TEMPERATURE = 0.0
SEED = 42
NUM_PER_CLASS = 1              # >1 only valid with STYLE="llm_multi"
OVERWRITE = False              # refuse to clobber an existing frozen file unless True
API_KEY = ""                   # Kaggle Secret (GEMINI_API_KEY) or pasted directly

In [ ]:
if not API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
        print("[auth] Loaded API_KEY from Kaggle Secret 'GEMINI_API_KEY'.")
    except Exception as exc:
        print(f"[auth] No Kaggle Secret 'GEMINI_API_KEY' found ({exc}).")
        print("       Paste a key into API_KEY above, or add a Kaggle Secret named")
        print("       GEMINI_API_KEY (Add-ons -> Secrets) before running this cell.")
else:
    print("[auth] Using API_KEY from the EDIT cell.")

assert API_KEY, "No API key available -- set API_KEY or a Kaggle Secret GEMINI_API_KEY"

In [ ]:
import json

import yaml

from features.descriptions import description_path, generate_descriptions

In [ ]:
assert isinstance(DATASET, str) and isinstance(STYLE, str), (
    "DATASET/STYLE are single values, not lists -- re-run the notebook to sweep (\u00a72.0)"
)
assert not MODEL.startswith("gemini-3"), (
    f"model={MODEL!r}: temperature/topK/topP are deprecated and silently ignored on "
    "the gemini-3.x family (3.7-flash, 3.6-flash, 3.5-flash-lite) -- generating with "
    "temperature=0.0 against one of these is a silent no-op, not an error. Use a "
    "gemini-2.x model."
)
if STYLE != "llm_multi":
    assert NUM_PER_CLASS == 1, f"NUM_PER_CLASS={NUM_PER_CLASS} only applies to STYLE='llm_multi'"

with open("config/config.yaml", "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)
dataset_info = config["datasets"][DATASET]
class_names = list(dataset_info["descriptions"])

out_path = Path(description_path(DATASET, STYLE))
if out_path.is_file() and not OVERWRITE:
    raise FileExistsError(
        f"{out_path} already exists -- this is a frozen artifact, committed to the repo. "
        "Set OVERWRITE=True to regenerate deliberately (this will change the file's "
        "sha256 and invalidate every cached text prototype built from the old text)."
    )

print(f"dataset: {DATASET} ({len(class_names)} classes) | model: {MODEL} | style: {STYLE}")
print(f"classes: {class_names}")
print(f"output: {out_path}")
print()
print("NOTE: a hosted model call is not bit-for-bit reproducible even at temperature=0.0.")
print("      The frozen JSON file this cell writes is the reproducible artifact -- not")
print("      the act of calling the API. Re-running this notebook later may produce")
print("      different text even with identical settings.")

In [ ]:
payload = generate_descriptions(
    dataset=DATASET,
    style=STYLE,
    class_names=class_names,
    model=MODEL,
    temperature=TEMPERATURE,
    seed=SEED,
    num_per_class=NUM_PER_CLASS,
    api_key=API_KEY,
)

for name, text in payload["descriptions"].items():
    print(f"[{name}]")
    print(f"  {text}")
print()
print(f"sha256: {payload['sha256']}")

In [ ]:
# Verify before writing: every class has a non-empty description (or, for
# llm_multi, NUM_PER_CLASS non-empty variants), and the class order matches
# config.yaml exactly -- the same order extract_vlm_features.ipynb and every
# downstream reader assumes.
assert list(payload["descriptions"]) == class_names, "class order drifted from config.yaml"
for name, text in payload["descriptions"].items():
    if STYLE == "llm_multi":
        assert isinstance(text, list) and len(text) == NUM_PER_CLASS, (
            f"{name}: expected {NUM_PER_CLASS} variants, got {text!r}"
        )
        assert all(isinstance(v, str) and v.strip() for v in text), f"{name}: empty variant"
    else:
        assert isinstance(text, str) and text.strip(), f"{name}: empty description"
print("OK -- all classes have non-empty descriptions in the correct order.")

In [ ]:
# No archive/zip: this is a small JSON file, committed straight to the repo
# (unlike every other notebook here, which produces GB-scale caches that only
# fit through Kaggle's Output tab as a zip).
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", encoding="utf-8") as handle:
    json.dump(payload, handle, indent=2, sort_keys=True, ensure_ascii=False)

print(f"Wrote {out_path} ({out_path.stat().st_size} bytes)")
print()
print("NEXT STEPS")
print(f"  1. Download {out_path} from the Kaggle Output tab (or copy its contents).")
print(f"  2. Commit it to the repo at the same path: {out_path}")
print("  3. extract_vlm_features.ipynb with DESCRIPTION_STYLE=" + repr(STYLE) +
      " will then find it.")